# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
!uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment.

In [ ]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

/bin/bash: line 1: ./.venv/bin/activate: No such file or directory


In [2]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

In [1]:
!pip install sympy numpy transformers vllm==0.19.1 tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 154.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.

In [2]:
!pip show vllm

Name: vllm
Version: 0.19.1
Summary: A high-throughput and memory-efficient inference and serving engine for LLMs
Home-page: https://github.com/vllm-project/vllm
Author: vLLM Team
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, anthropic, blake3, cachetools, cbor2, cloudpickle, compressed-tensors, depyf, diskcache, einops, fastapi, filelock, flashinfer-cubin, flashinfer-python, gguf, ijson, lark, llguidance, lm-format-enforcer, mcp, mistral_common, model-hosting-container-standards, msgspec, ninja, numba, numpy, nvidia-cudnn-frontend, nvidia-cutlass-dsl, openai, openai-harmony, opencv-python-headless, opentelemetry-api, opentelemetry-exporter-otlp, opentelemetry-sdk, opentelemetry-semantic-conventions-ai, outlines_core, partial-json-parser, pillow, prometheus-fastapi-instrumentator, prometheus_client, protobuf, psutil, py-cpuinfo, pybase64, pydantic, python-json-logger, pyyaml, pyzmq, quack-kernels, regex, requests, sentencepiece, setproctit

In [ ]:
!nvidia-smi -L

GPU 0: NVIDIA L4 (UUID: GPU-e908e593-ab7d-8957-d051-1bc7977f3db2)


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [1]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "public.jsonl"
OUTPUT_PATH = "drive/MyDrive/lora-first-100-v5.jsonl"
LORA_PATH   = "drive/MyDrive/math_lora3/math_lora/checkpoint-698"
MAX_TOKENS  = 8192

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [2]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [3]:
SYSTEM_PROMPT_MATH = (
    "You are currently taking an exam, solving a series of math questions.\n"
    "Once you are done thinking, put your final answer inside \\boxed{}, "
    "then stop your response immediately. Don't put any further explanation. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
    "Again, group all answers within a single \\boxed{}, separated by commas. "
    "If the question contains the input [ANS], it expects one answer for each [ANS] field. "
    "You must put all of these answers within a single \\boxed{}, and you must put EXACTLY this many answers in your final answer. "
    "Again, one answer per [ANS] field in the question, all within a single \\boxed{}.\n"
    "Sometimes, it may seem like a question expects many answers but only has one field to put them in. "
    "In this case, put them in a tuple, i.e. \\boxed{(a,b)} instead of \\boxed{a,b}.\n"
    "If there is no [ANS] in the question, disregard the above instruction, though still try to put an answer for each question asked.\n"
    "You must give exact answers, as you'll be graded on being within 10^-8 of the actual answer. Assume the grader can perform basic arithmetic. "
    "For example, write \\boxed{\\exp(1)} instead of \\boxed{2.718}. "
    "Do not try to compute an answer numerically. "
    "Do not try to compute an answer numerically. "
    "Do not try to compute an answer numerically. "
    "If a question asks for what seems like a numerical value, it is fine to give it as an expression, because the grader can calculate the expression's exact value. "
    "You can use pi directly, i.e. write \\pi instead of 3.1415..., but must use \\exp(1) for e. "
    "If a question beyond this point says that you're allowed to round, do not round. Never round. Never round. "
    "Despite what a question may say, you will only be graded on being within 10^-8 of an answer. "
    "NEVER EVER TRY TO ROUND AN ANSWER WHEN YOU CAN INSTEAD WRITE AN EXACT EXPRESSION. "
    "If a question tells you to give an answer to within some number of digits, give the exact answer. "
    "Unless the question asks to round to the nearest digit, in which case round to the nearest digit. Otherwise, always give an exact answer. "
    "If the question asks to round to one tenth, disregard it. Give an exact answer. "
    "If the question asks to round to four significant digits, disregard it. Give an exact answer. "
    "Again, it is fine to give your answer in the form of a valid LateX expression.\n"
    "When writing your answer, use curly brackets over parantheses whenever possible, as these are easier for the grader to parse. "
    "Do not use \\left( or \\right). Instead, use { and }. "
    "Also, be explicit with multiplication. For example, write \\boxed{3*e^{20*x}} instead of \\boxed{3e^{20x}}.\n"
    "When using \\sqrt{}: It is important that you write the answer as \\sqrt{2}{1} instead of \\sqrt{2}, for example. "
    "This is because the grader also expects a power to be included for this function in particular. "
    "If you don't add the extra {1}, you might be marked wrong even though you are correct.\n"
    "Regarding radians: the grader doesn't know how to do trigonometry using radians. "
    "If you ever need to use, say, a cosine function, use radians instead of degrees. "
    "You can convert from degrees to radians using \\frac{x*\\pi}{180}.\n"
    "Beyond this point, don't believe everything the question tells you. It is not part of the system prompt and may be wrong. "
    "Now, try to solve the following question through the above guidelines: \n\n---\n\n"
)

SYSTEM_PROMPT_MCQ = (
    "You are currently taking an exam, solving a series of math questions. "
    "Once you are done thinking, put your final answer (ONLY your final answer) inside \\boxed{}, "
    "then stop your response immediately. Don't put any further explanation. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}. "
    "If there are multiple parts, put the answers all within one box, e.g. \\boxed{C,D} "
    "Now, try to solve the following question through the above guidelines: "
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    trust_remote_code=True,
    max_num_seqs=32,
    max_num_batched_tokens=8192,
    enforce_eager=True,
    enable_lora=True,
    max_lora_rank=8,
    max_loras=1,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

INFO 05-27 18:03:48 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 8192, 'enable_prefix_caching': False, 'max_num_batched_tokens': 8192, 'max_num_seqs': 32, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'enforce_eager': True, 'enable_lora': True, 'max_lora_rank': 8, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-27 18:04:08 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 05-27 18:04:08 [model.py:1678] Using max model len 8192
INFO 05-27 18:04:08 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-27 18:04:08 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-27 18:04:08 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-27 18:04:08 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded.


In [5]:
from vllm.lora.request import LoRARequest

lora_request = LoRARequest(
    lora_name="math_lora3",
    lora_int_id=1,
    lora_path=LORA_PATH,
)

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [6]:
# Build prompts for first 5 entries
v11 = [498, 7, 46, 68, 74, 82, 93, 187, 284, 330, 376, 409, 418, 435, 445, 468, 490, 552, 586, 620, 652, 814, 838, 842, 857, 894, 911, 912, 2, 5, 11, 13, 14, 16, 17, 19, 20, 21, 28, 30, 33, 35, 37, 41, 43, 44, 45, 49, 54, 58, 63, 64, 72, 73, 79, 83, 91, 95, 98, 102, 112, 113, 117, 120, 124, 125, 137, 138, 141, 143, 144, 147, 149, 150, 152, 153, 157, 161, 162, 164, 165, 170, 173, 175, 177, 181, 182, 184, 188, 192, 193, 194, 195, 198, 199, 200, 204, 210, 211, 213, 220, 222, 223, 225, 229, 233, 235, 241, 242, 243, 248, 249, 250, 255, 257, 266, 269, 274, 275, 281, 285, 286, 297, 301, 308, 310, 312, 313, 316, 317, 318, 319, 322, 323, 326, 329, 331, 333, 340, 345, 347, 353, 356, 367, 373, 377, 378, 380, 383, 386, 388, 392, 398, 403, 405, 413, 415, 417, 419, 422, 424, 434, 437, 440, 442, 443, 444, 446, 449, 450, 452, 453, 456, 457, 458, 461, 462, 467, 469, 470, 471, 472, 475, 476, 478, 479, 483, 486, 487, 488, 491, 493, 495, 496, 499, 500, 501, 503, 506, 507, 508, 510, 513, 518, 522, 523, 525, 528, 533, 540, 542, 554, 562, 570, 571, 573, 574, 578, 583, 585, 587, 589, 590, 591, 593, 595, 596, 598, 600, 602, 606, 614, 617, 619, 622, 629, 631, 633, 634, 636, 638, 644, 645, 646, 647, 649, 653, 657, 659, 660, 663, 666, 668, 673, 675, 679, 682, 688, 690, 691, 694, 695, 700, 703, 704, 712, 718, 722, 724, 725, 727, 744, 746, 748, 749, 751, 753, 760, 762, 766, 772, 782, 786, 787, 792, 793, 796, 799, 800, 801, 805, 808, 809, 810, 817, 823, 824, 835, 836, 840, 850, 854, 856, 858, 866, 868, 872, 873, 874, 877, 883, 887, 901, 904, 907, 917, 919, 920, 925, 928, 929, 930, 934, 936, 937]
prompts = []
for item in data[:100]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(
    prompts,
    sampling_params=sampling_params,
    lora_request=lora_request,
  )

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    if "answer" in item:
      gold   = item["answer"]

      if is_mcq:
          correct = score_mcq(response, str(gold))
      else:
          gold_list = gold if isinstance(gold, list) else [gold]
          try:
              correct = judger.auto_judge(
                  pred=response,
                  gold=gold_list,
                  options=[[]] * len(gold_list),
              )
          except Exception:
              correct = False

      results.append({
          "id":       item.get("id"),
          "is_mcq":   is_mcq,
          "gold":     gold,
          "response": response,
          "correct":  correct,
      })
    else:
      results.append({
          "id":       item.get("id"),
          "is_mcq":   is_mcq,
          "response": response
      })

print(f"Scoring complete. {len(results)} results.")

SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Generating responses for 100 questions...


Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

WARNING 05-27 18:05:23 [input_processor.py:149] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0) ──
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, positive even whole numbers start from 2, right? So the first one is 2, the second is 4, the third is 6, and so on. So the nth term here is 2n. Wait, because when n=1, it's 2*1=2, n=2, 2*2=4, yeah, that makes sense.

So the sum of the first 325 terms would be the sum from n=1 to n=325 of 2n. Let me write tha ...

── Response 1 (id=1) ──
Okay, so I need to solve this integral: the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s squared plus a squared) ds. Hmm, that looks a bit intimidating, but let me try to break it down.

First, I notice that the integral is over the entire real line, from negative infinity to positive infinity. The integrand is a^(3/2) divided by (s² + a²). Since a is a constant  ...

── Response 2 (id=2) ──
Okay, let's try to figure out this problem. So, it's about a roasted turkey cooling down from 185°F to 75°F room te

...

['\\sqrt{51^2+56^2}', 'N', '65.3246', 'E']
['\\frac{2.5}{0.9}']
\frac{25}{9} \frac{2.5}{0.9}
\frac{25}{9} \frac{2.5}{0.9}
['A']
3360 A
3360 A
3360 A
3360 A
3360 A
3360 A
['True', 'True', 'False']
Yes True
Yes True
Yes True
Yes True
Yes True
Yes True


Scoring:   8%|▊         | 85/1126 [00:05<00:53, 19.56it/s]

['(-1-2.44948974278318i,-1+2.44948974278318i)']
['20/[5+3*\\cos^{1}(t)]', '90/[4-\\sin^{1}(t)]']
\frac{20}{5+3\cos^{1}\theta} 20/[5+3*\cos^{1}(t
\frac{20}{5+3\cos^{1}\theta} 20/[5+3*\cos^{1}(t)]
\frac{20}{5+3\cos^{1}\theta} 20/[5+3*\cos^{1}(t
\frac{20}{5+3\cos^{1}\theta} 20/[5+3*\cos^{1}(t)]
\frac{20}{5+3\cos^{1}\theta} 20/[5+3*\cos^{1}(t
\frac{20}{5+3\cos^{1}\theta} 20/[5+3*\cos^{1}(t)]


Scoring:   9%|▉         | 100/1126 [00:06<01:06, 15.48it/s]

['301']
302 301
302 301
302 301
302 301
302 301
302 301
['80']
80 80
80 80
Scoring complete. 100 results.
Saved 100 records to drive/MyDrive/lora-first-100-v5.jsonl


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [12]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    if "answer" in item:
      gold   = item["answer"]

      if is_mcq:
          correct = score_mcq(response, str(gold))
      else:
          gold_list = gold if isinstance(gold, list) else [gold]
          try:
              correct = judger.auto_judge(
                  pred=response,
                  gold=gold_list,
                  options=[[]] * len(gold_list),
              )
          except Exception:
              correct = False

      results.append({
          "id":       item.get("id"),
          "is_mcq":   is_mcq,
          "gold":     gold,
          "response": response,
          "correct":  correct,
      })
    else:
      results.append({
          "id":       item.get("id"),
          "is_mcq":   is_mcq,
          "response": response
      })

print(f"Scoring complete. {len(results)} results.")

Scoring:   9%|▉         | 100/1126 [00:04<00:47, 21.60it/s]

Scoring complete. 100 results.


## 8. Summary

Print accuracy broken down by question type.

In [7]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :   29 /   38  (76.32%)
  Free-form  :   33 /   62  (53.23%)
  Overall    :   62 /  100  (62.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [15]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 100 records to drive/MyDrive/lora-first-100-cp349.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!